In [ ]:
import pandas as pd 
import glob 
import numpy as np
import os

In [ ]:
df = pd.read_csv("../../data/raw/crime_monthly/Geregistreerde misdrijven month 1.csv", sep=";"
)
df.head()  




In [ ]:
df

In [ ]:
path = "../../data/raw/crime_monthly/"
all_misdrijf_files = sorted(glob.glob(path + "*.csv"))

dfs = []

for file in all_misdrijf_files:
    df = pd.read_csv(file, sep=None, engine="python")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True) 
df_all

In [ ]:
df_crime_average_per_year = df_all.groupby('\ufeffWijk')['Totaal misdrijven'].mean().reset_index() 
df_crime_average_per_year.rename(columns={'\ufeffWijk':'Wijk'}, inplace=True) 
df_crime_average_per_year['Totaal misdrijven'] = df_crime_average_per_year['Totaal misdrijven'].round(0)  
df_crime_average_per_year

ik ga nu de tabel ff helemaal goed zetten en fixen  
(working from the intermediate total table assembled earlier in the project)


In [ ]:
df = pd.read_csv("../../data/intermediate/total_table.csv")
df

I cleaned the data set of CBS wijken data with the amount of inwoners and oppervlakte per wijk, to make it ready to merge to the final data frame

In [ ]:
path = "../../data/raw/Kerncijfers_2023.csv"  

# 1) find start data
with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()
start = next(i for i, line in enumerate(lines) if line.strip().startswith('"Onderwerp"'))

# 2)read csv from that row
raw = pd.read_csv(
    path,
    sep=";",
    skiprows=start,
    quotechar='"',
    encoding="utf-8",
    low_memory=False
).set_index("Onderwerp")

# 3) Take the rows needed + transpose
needed = raw.loc[
    [
        "Regioaanduiding|Codering",
        "Regioaanduiding|Soort regio",
        "Bevolking|Aantal inwoners",
        "Oppervlakte|Oppervlakte totaal",
    ]
].T

# 4) rename columns
needed = needed.rename(columns={
    "Regioaanduiding|Codering": "code",
    "Regioaanduiding|Soort regio": "soort_regio",   # Gemeente / Wijk / Buurt
    "Bevolking|Aantal inwoners": "inwoners",
    "Oppervlakte|Oppervlakte totaal": "oppervlakte_ha",
})

# 5) clean
needed["inwoners"] = (
    needed["inwoners"].astype(str).str.replace(" ", "", regex=False)
)
needed["inwoners"] = pd.to_numeric(needed["inwoners"], errors="coerce").astype("Int64")
needed["oppervlakte_ha"] = pd.to_numeric(needed["oppervlakte_ha"], errors="coerce")

# km² oppervlakte
needed["oppervlakte_km2"] = needed["oppervlakte_ha"] / 100

df_final = needed[["code", "soort_regio", "inwoners", "oppervlakte_ha", "oppervlakte_km2"]].dropna(subset=["code"])

# 6) only  wijken (drop buurten + gemeenten)
df_wijken = df_final[df_final["soort_regio"].str.strip() == "Wijk"].copy()

df_wijken

Merged the Wijk info to the final data frame

In [ ]:
# 0) code cleaning
df["wijkcode"] = df["wijkcode"].astype(str).str.strip().str.upper()
df_wijken["code"] = df_wijken["code"].astype(str).str.strip().str.upper()

# 1) Select columns to merge
cbs_cols = df_wijken[["code", "inwoners", "oppervlakte_ha", "oppervlakte_km2"]].copy()

# 2) Merge: alleen wijken die al in final_df staan blijven 
final_df = df.merge(
    cbs_cols,
    left_on="wijkcode",
    right_on="code",
    how="left"
)

# 3) 'code' droppen anders dubbel
final_df= final_df.drop(columns=["code"])

final_df

Merged the boom per wijk data to the final data frame 

In [ ]:
bomen_per_wijk = pd.read_csv("../../data/intermediate/bomen_per_wijk.csv")  
bomen_per_wijk.head()

In [ ]:
final_df2 = final_df.merge(
    bomen_per_wijk[["wijkcode", "aantal_bomen"]],
    on="wijkcode",
    how="left"
)


final_df2.head()

Transferred all columns to rates, so that it does not depend on inwoners or size anymore

In [ ]:
final_df2['Misdrijven_per_1000_inwoners'] = (final_df2['Totaal misdrijven'] / final_df2['inwoners']) * 1000   
final_df2['Lights_per_km2'] = final_df2['aantal_lantaarns'] / final_df2['oppervlakte_km2']  
final_df2['Trees_per_km2'] = final_df2['aantal_bomen'] / final_df2['oppervlakte_km2'] 


Alle afstanden tot data gegroepeerd op intuitieve groepen 
(deze moeten we nog ff onderbouwen met literatuur/ ergens op baseren)

In [ ]:
#Made the groups  
groups = {
    # Uitgaan / avond-economie (crime attractors)
    "nightlife": [
        "AfstandTotCafeED_36",
        "AfstandTotCafetariaED_40",
        "AfstandTotRestaurant_44",
        "AfstandTotPoppodium_103",
        "AfstandTotBioscoop_104",
        "AfstandTotHotelED_48",   # hotel vaak overlap met toerisme/uitgaan
    ],

    # Mobiliteit / bereikbaarheid (access + doorstroom)
    "mobility": [
        "AfstandTotOpritHoofdverkeersweg_89",
        "AfstandTotTreinstationsTotaal_90",
        "AfstandTotBelangrijkOverstapstation_91",
    ],

    # Retail / dagelijkse drukte (targets + passanten)
    "retail_daily": [
        "AfstandTotGroteSupermarkt_24",
        "AfstandTotOvDagelLevensmiddelen_28",
        "AfstandTotWarenhuis_32",
    ],

    # Onderwijs / jeugd-routine (routine peaks)
    "education_youth": [
        "AfstandTotKinderdagverblijf_52",
        "AfstandTotBuitenschoolseOpvang_56",
        "AfstandTotSchool_60",
        "AfstandTotSchool_64",
        "AfstandTotSchool_68",
        "AfstandTotSchool_72",
    ],

    # Cultuur / sport / publieke voorzieningen (events + bezoekers)
    "culture_sport_public": [
        "AfstandTotBibliotheek_92",
        "AfstandTotZwembad_93",
        "AfstandTotKunstijsbaan_94",
        "AfstandTotMuseum_95",
        "AfstandTotPodiumkunstenTotaal_99",
        "AfstandTotAttractie_110",
    ],



    # Institutioneel / emergency/health (meestal control/proxy)
    "emergency_health": [
        "AfstandTotBrandweerkazerne_114",
        "AfstandTotHuisartsenpraktijk_5",
        "AfstandTotHuisartsenpost_9",
        "AfstandTotApotheek_10",
        "AfstandTotZiekenhuis_11",
        "AfstandTotZiekenhuis_15",
    ],
}

#min&mean distance features per group Ik heb nu beide toegevoegd want weet niet persee welke beter werkt, of het mean is of min afstand tot een station bijv significanter is.

def add_group_distance_features(df: pd.DataFrame, groups: dict) -> pd.DataFrame:
    df = df.copy()

    #check op ontbrekende kolommen
    missing = {g: [c for c in cols if c not in df.columns] for g, cols in groups.items()}
    missing = {g: cols for g, cols in missing.items() if cols}
    if missing:
        print("Let op: deze kolommen ontbreken in je df:")
        for g, cols in missing.items():
            print(f" - {g}: {cols}")

    for g, cols in groups.items():
        cols_present = [c for c in cols if c in df.columns]
        if not cols_present:
            continue

        df[f"min_dist_{g}"] = df[cols_present].min(axis=1)
        df[f"mean_dist_{g}"] = df[cols_present].mean(axis=1)

    return df

final_df_feat = add_group_distance_features(final_df2, groups) 


# Maak 2 feature-sets: alleen min vs alleen mean
min_cols  = [c for c in final_df_feat.columns if c.startswith("min_dist_")]
mean_cols = [c for c in final_df_feat.columns if c.startswith("mean_dist_")]

print("Min features:", min_cols)
print("Mean features:", mean_cols) 

total_final_df

In [ ]:
total_final_df.to_csv("../../data/intermediate/total_final_df.csv", index=False)

Starting with exploratory analysis

In [ ]:
total_final_df.columns

In [ ]:
#check for is null values --> No missing values
total_final_df.isnull().sum().sum()

In [ ]:
#checked if there are duplicates
total_final_df["wijkcode"].nunique()

Data set is clean so now I checked what the data looks like: 
Starting with Crime: 


#### Crimes distribution

In [ ]:
total_final_df[["Misdrijven_per_1000_inwoners"]].describe() 
total_final_df[["inwoners"]].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(total_final_df["Misdrijven_per_1000_inwoners"], bins=30)
plt.xlabel("Crimes per 1,000 inhabitants (2023)")
plt.ylabel("Number of neighbourhoods")
plt.title("Distribution of crime rate (2023)")
plt.show()  #crimes heel raar verdeeld onderzoeken

plt.figure()
plt.hist(total_final_df["inwoners"], bins=30)
plt.xlabel("population (2023)")
plt.ylabel("Number of neighbourhoods")
plt.title("population (2023)")
plt.show()  

plt.figure()
plt.hist(total_final_df["Totaal misdrijven"], bins=30)
plt.xlabel("count of misdrijven (2023)")
plt.ylabel("Number of neighbourhoods")
plt.title("Distribution of crime count (2023)")
plt.show() 

In [ ]:
plt.figure()
plt.boxplot(total_final_df["Misdrijven_per_1000_inwoners"], vert=False)
plt.xlabel("Crimes per 1,000 inhabitants (2023)")
plt.title("Crime rate outliers (2023)")
plt.show() 

plt.figure()
plt.boxplot(total_final_df["Totaal misdrijven"], vert=False)
plt.xlabel("Count of crimes (2023)")
plt.title("Crime count outliers (2023)")
plt.show()

In [ ]:
total_final_df.sort_values("Misdrijven_per_1000_inwoners", ascending=False)[
    ["wijkcode", "Misdrijven_per_1000_inwoners", "inwoners", "Totaal misdrijven", 'wijknaam']
].head(50) #so wierd boxplot caused by small populations causing huge ratios, so we have to do in the sensitivity analysis maybe  
# a version with only wijken with more than 1000 inhibitants

In [ ]:
total_final_df['inwoners'].describe()

In [ ]:
plt.figure()
plt.boxplot(np.log1p(total_final_df["Misdrijven_per_1000_inwoners"]), vert=False) #now the big ratios are less extreme
plt.xlabel("Crimes per 1,000 inhabitants (2023)")
plt.title("Crime rate outliers (2023)")
plt.show() 

“The crime rate (crimes per 1,000 residents) shows a strongly right-skewed distribution with a small number of extreme outliers. Inspection of the highest-rate districts reveals that these outliers are driven by very small resident populations (e.g., 21 crimes among 35 residents). This ‘small numbers’ issue makes rate estimates unstable for low-population districts, which we account for in subsequent analysis by using a log(1+rate) transformation / or excluding wijken below a population threshold in a robustness check).” VRAAG> WAT IS BETER 

#### Trees & lights distribution

In [ ]:
feature_cols = ['Lights_per_km2', 'Trees_per_km2']

total_final_df[feature_cols].describe().T

In [ ]:
for col in feature_cols:
    x = total_final_df[col].dropna()

    plt.figure()
    plt.hist(x, bins=30)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"Distribution of {col}")
    plt.show()

    plt.figure()
    plt.boxplot(x, vert=False)
    plt.xlabel(col)
    plt.title(f"Boxplot of {col}")
    plt.show()

#### Distances to

In [ ]:
#groups zijn al sterk onderbouwt op logica nog ff checken hoe correlatie is
target = "Misdrijven_per_1000_inwoners"

group_corr_summary = []

for gname, cols in groups.items():
    X = total_final_df[cols].copy()

    
    if len(cols) >= 2:
        c = X.corr(method="spearman")   #spearman = robuuster bij non-lineair
        # gemiddelde abs correlatie zonder diagonaal
        mean_abs_corr = (c.abs().values.sum() - len(cols)) / (len(cols)*(len(cols)-1))
    else:
        mean_abs_corr = None

    group_corr_summary.append({
        "group": gname,
        "n_vars": len(cols),
        "mean_abs_spearman_within_group": None if mean_abs_corr is None else round(mean_abs_corr, 3)
    })

pd.DataFrame(group_corr_summary).sort_values("mean_abs_spearman_within_group", ascending=False) 

#0.4-0.7 = nice choerent group, rest is more broad 
#so overall nice coherent groups, others less but meaningfull

To validate our facility groupings, we computed the mean absolute Spearman correlation among the distance variables within each group. Nightlife and retail groups show high within-group coherence (0.74 and 0.63), while mobility is lower (0.23), indicating it captures a broader, multi-faceted accessibility dimension

In [ ]:
min_cols  = [f"min_dist_{g}"  for g in groups.keys()]
mean_cols = [f"mean_dist_{g}" for g in groups.keys()]

rows = []
for g, cmin, cmean in zip(groups.keys(), min_cols, mean_cols):
    rows.append({
        "group": g,
        "spearman_min_vs_crime": total_final_df[cmin].corr(total_final_df[target], method="spearman"),
        "spearman_mean_vs_crime": total_final_df[cmean].corr(total_final_df[target], method="spearman"),
        "corr_min_vs_mean": total_final_df[cmin].corr(total_final_df[cmean], method="spearman"),
    })

comp = pd.DataFrame(rows)
comp[["spearman_min_vs_crime","spearman_mean_vs_crime","corr_min_vs_mean"]] = comp[
    ["spearman_min_vs_crime","spearman_mean_vs_crime","corr_min_vs_mean"]
].round(3)

comp 

#lees hier de richting van de relatie dus distance nightlife omlaag = crime omhoog, en hoe hoger hoe sterker  
#corr onderling tussen mean en min if high number they tell almost the same otherwise different so choice is important 

#moet dit nog specefieker? mag ik nu mean kiezen voor consistentie en over het algemeen beter

Seeing what the relationships with crime look like

In [ ]:
#data to work with
target = "Misdrijven_per_1000_inwoners"

facility_mean_cols = [
    "mean_dist_nightlife",
    "mean_dist_mobility",
    "mean_dist_retail_daily",
    "mean_dist_education_youth",
    "mean_dist_culture_sport_public",
    "mean_dist_emergency_health",
]

feature_cols = ["Trees_per_km2", "Lights_per_km2"] + facility_mean_cols

df_model = total_final_df[["wijkcode", target] + feature_cols].dropna().copy()


df_model["log_crime_rate"] = np.log1p(df_model[target]) 
df_model.head()

In [ ]:
corr = pd.DataFrame({
    "spearman_vs_crime": [df_model[c].corr(df_model[target], method="spearman") for c in feature_cols],
    "pearson_vs_crime":  [df_model[c].corr(df_model[target], method="pearson")  for c in feature_cols],
}, index=feature_cols).round(3).sort_values("spearman_vs_crime")

corr 

#pearson zegt of het rechte lijn verband heeft, spearman of het rangordeverband (of het onver algemeen ook groter wordt) heeft (als ze heel anders zijn dan vaak geen lineair verband (hoe hoger hoe beter))

In [ ]:
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix

# kies kolommen die je wil plotten
cols = [
    "log_crime_rate",
    "Trees_per_km2",
    "Lights_per_km2",
    "mean_dist_nightlife",
    "mean_dist_mobility",
    "mean_dist_retail_daily",
    "mean_dist_education_youth",
    "mean_dist_culture_sport_public",
    "mean_dist_emergency_health",
]

# mapping van oude -> nieuwe namen
rename_map = {
    "log_crime_rate": "crime_rate_log",
    "Trees_per_km2": "trees",
    "Lights_per_km2": "lights",
    "mean_dist_nightlife": "nightlife",
    "mean_dist_mobility": "mobility",
    "mean_dist_retail_daily": "retail_daily",
    "mean_dist_education_youth": "education_youth",
    "mean_dist_culture_sport_public": "culture_sport",
    "mean_dist_emergency_health": "emergency_health",
}

df_plot = df_model[cols].dropna().rename(columns=rename_map).copy()

# (optioneel) filter om extreme rate outliers te verminderen
# df_plot = total_final_df[total_final_df["inwoners"] > 500][cols].dropna().rename(columns=rename_map).copy()

axes = scatter_matrix(df_plot, figsize=(18, 18), alpha=0.5, diagonal="hist")

# labels leesbaar maken
for ax in axes.flatten():
    ax.xaxis.label.set_rotation(45)
    ax.yaxis.label.set_rotation(0)
    ax.xaxis.label.set_fontsize(9)
    ax.yaxis.label.set_fontsize(9)

plt.tight_layout()
plt.show()


In [ ]:
df_plot.head()

In [ ]:
#deze is het zelfde als vorige maar fancyer
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



# 4) Seaborn: fancy pairplot met regressielijn + CI band
sns.set_theme(style="whitegrid")

g = sns.pairplot(
    df_plot,
    kind="reg",                 # regressielijn in off-diagonal
    diag_kind="kde",            # mooie diagonaal (kan ook "hist")
    plot_kws={"scatter_kws": {"s": 20, "alpha": 0.7},
              "line_kws": {"linewidth": 2}},
    height=2.0,
    corner=False                # True = alleen lower triangle (minder druk)
)

g.fig.suptitle("Pairplot with regression lines (crime_rate_log + features)", y=1.02)
plt.show()


A scatterplot matrix of the log-transformed crime rate and design features shows that proximity to nightlife, retail, and cultural amenities is associated with higher crime rates (negative relationship between distance and crime). Tree density shows a weaker negative association with crime, while streetlight density exhibits little clear monotone pattern. Several accessibility variables are strongly correlated with each other, indicating that central neighbourhoods tend to be close to multiple amenities, which may introduce collinearity in regression-based models

In [ ]:
#creating final data frame for machine learning: 
total_final_df.columns


In [ ]:
#inwoners zorgde voor outliers, small populations neighbourhoods caused huge ratios: so i searched what treshold would make sense; 

plt.figure()
plt.scatter(total_final_df["inwoners"], total_final_df["Misdrijven_per_1000_inwoners"], alpha=0.7)
plt.xlabel("Inwoners")
plt.ylabel("Crimes per 1,000 inhabitants")
plt.title("Crime rate vs population (check instability at low n)")
plt.tight_layout()
plt.show() 
#dit moet in report 
#je ziet hier inderdaad dat de enorm hoge crime rates bij hele lage inwoners zitten, maar dat dit er eigenlijk maar 1 is,  
#als je de treshold (dus zegt nemen alleen populaties >500 mee) hoog maakt verlies je eigenlijk data, maar het voegt niks toe want  
#die grote outlier zit al bij 35 inwoners. Daarnaast blijft er wel een hoge tail in zitten maar niet persee bij hele lage inwoners, 
#dus de rest compenseer je met log transform, want de andere outliers horen (bijv red light district) dus hoeft niet weg

In [ ]:
#dus wat er in report

threshold = 50

X_cols = [
    "Trees_per_km2",
    "Lights_per_km2",
    "mean_dist_nightlife",
    "mean_dist_mobility",
    "mean_dist_retail_daily",
    "mean_dist_education_youth",
    "mean_dist_culture_sport_public",
    "mean_dist_emergency_health",
]

df_final_model = total_final_df[total_final_df["inwoners"] >= threshold].copy()
df_final_model["crime_rate_log"] = np.log1p(df_model["Misdrijven_per_1000_inwoners"])

df_final_model = df_final_model(subset=["crime_rate_log"] + X_cols).copy()

df_model.shape


In [ ]:
import numpy as np
import pandas as pd

threshold = 50

# originele kolomnamen (zoals in total_final_df)
X_cols = [
    "Trees_per_km2",
    "Lights_per_km2",
    "mean_dist_nightlife",
    "mean_dist_mobility",
    "mean_dist_retail_daily",
    "mean_dist_education_youth",
    "mean_dist_culture_sport_public",
    "mean_dist_emergency_health",
]

# 1) Filter op inwoners
df_final_model = total_final_df.loc[total_final_df["inwoners"] >= threshold].copy()

# 2) Maak log target
df_final_model["crime_rate_log"] = np.log1p(df_final_model["Misdrijven_per_1000_inwoners"])

# 3) Houd alleen relevante kolommen
keep_cols = [
    "wijkcode",
    "wijknaam",
    "inwoners",
    "Misdrijven_per_1000_inwoners",
    "crime_rate_log",
] + X_cols

df_final_model = df_final_model[keep_cols].copy()

# (optioneel) als je echt zeker wil zijn dat modelkolommen geen NaN hebben:
# df_final_model = df_final_model.dropna(subset=["crime_rate_log"] + X_cols).copy()

# 4) Rename naar Engels
rename_map = {
    "wijkcode": "neighbourhood_code",
    "wijknaam": "neighbourhood_name",
    "inwoners": "population",
    "Misdrijven_per_1000_inwoners": "crime_rate_per_1000",
    "crime_rate_log": "log_crime_rate",
    "Trees_per_km2": "tree_density_per_km2",
    "Lights_per_km2": "light_density_per_km2",
    "mean_dist_nightlife": "mean_dist_nightlife",
    "mean_dist_mobility": "mean_dist_mobility",
    "mean_dist_retail_daily": "mean_dist_retail",
    "mean_dist_education_youth": "mean_dist_education",
    "mean_dist_culture_sport_public": "mean_dist_culture_sport",
    "mean_dist_emergency_health": "mean_dist_emergency_health",
}

df_final_model = df_final_model.rename(columns=rename_map)

# 5) Engelse feature list (na rename!)
X_cols_en = [
    "tree_density_per_km2",
    "light_density_per_km2",
    "mean_dist_nightlife",
    "mean_dist_mobility",
    "mean_dist_retail",
    "mean_dist_education",
    "mean_dist_culture_sport",
    "mean_dist_emergency_health",
]

df_final_model.head()


In [ ]:
df_final_model.to_csv("../../data/final/df_final_model.csv", index=False)